# 08_phase3_composition_scCODA.ipynb
Phase 3 — Compositional Analysis (scCODA)

**Why scCODA instead of the Phase 2 Kruskal-Wallis pass:** cell-type proportions are compositional data — they sum to 100% within each sample, so cell types aren't statistically independent (if one goes up, something else must go down). Standard tests (Kruskal-Wallis, Mann-Whitney) don't account for this. scCODA uses a Bayesian Dirichlet-Multinomial model built specifically for compositional data, modelling every cell type's change relative to one chosen reference type.

**Reference cell type choice matters** — it should be a population unlikely to be the one driving the biological difference under test, since scCODA's results are always interpreted relative to it. Chosen deliberately per comparison below, not defaulted.

In [ ]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
from pathlib import Path

try:
    from sccoda.util import cell_composition_data as scc_dat
    from sccoda.util import comp_ana as scc_ana
    print("sccoda imported successfully")
except ImportError:
    print("sccoda not installed. Run: pip install sccoda")
    raise

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3_composition_sccoda"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3_composition_sccoda"

for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EXCLUDE_TYPES = ["Unassigned (n=28, doublet/mixed-identity artefact)",
                  "Mixed/stromal-contaminated (CD8+fibroblast signal)"]

print("Setup complete")

In [ ]:
# ----------------------------
# Cell 2 — Build sample x cell_type count table (scCODA input format)
# scCODA expects an AnnData where obs = samples (not cells!) and
# var = cell types, X = counts of each cell type per sample.
# ----------------------------
def build_composition_anndata(adata_obs, sample_col, condition_col, exclude_types=EXCLUDE_TYPES):
    df = adata_obs[~adata_obs["cell_type"].isin(exclude_types)].copy()
    counts = df.groupby([sample_col, "cell_type"], observed=True).size().unstack(fill_value=0)

    condition_lookup = df.drop_duplicates(sample_col).set_index(sample_col)[condition_col]
    sample_meta = pd.DataFrame({condition_col: condition_lookup})

    comp_adata = scc_dat.from_pandas(counts, covariate_columns=[])
    comp_adata.obs[condition_col] = sample_meta.loc[comp_adata.obs_names, condition_col].values

    return comp_adata

print("Composition AnnData builder ready")

In [ ]:
# ----------------------------
# Cell 3 — GSE114725: Tumour vs Normal
# Reference: T cells — the largest, most stable population, and not
# expected a priori to be THE cell type driving tumour/normal difference
# (macrophages are the stronger candidate for that, per DE/LIANA
# findings already established — using them as reference could bias
# results toward finding everything else "changed" relative to them).
# ----------------------------
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad", backed="r")
obs1 = adata1.obs[adata1.obs["tissue"].isin(["TUMOR", "NORMAL"])].copy()

comp_adata1 = build_composition_anndata(obs1, sample_col="patient", condition_col="tissue")
print(f"GSE114725 composition data: {comp_adata1.n_obs} samples x {comp_adata1.n_vars} cell types")
print(comp_adata1.obs)

model1 = scc_ana.CompositionalAnalysis(
    comp_adata1, formula="tissue", reference_cell_type="T cells"
)
result1 = model1.sample_hmc()
print("\n" + "="*60)
print("GSE114725: Tumour vs Normal — scCODA results")
print("="*60)
result1.summary()

In [ ]:
# ----------------------------
# Cell 4 — GSE114725: extract and save credible effects
# scCODA reports "credible" effects (analogous to significant, but via
# Bayesian HDI/FDR rather than a classic p-value) — cell types whose
# change is confidently non-zero relative to the reference.
# ----------------------------
effects1 = result1.effect_df
effects1.to_csv(RESULTS_DIR / "GSE114725_scCODA_tumor_vs_normal_effects.csv")
print(effects1)

credible1 = result1.credible_effects()
print("\nCredible (confidently non-zero) effects:")
print(credible1)

In [ ]:
# ----------------------------
# Cell 5 — GSE176078: pairwise subtype comparisons
# Reference: PVL (perivascular-like) — a structural/stromal population,
# not expected to be a primary driver of subtype-specific immune or
# epithelial differences (unlike CAFs or Macrophages, which showed real
# DE/pathway signal already and would be riskier reference choices).
# ----------------------------
adata2 = sc.read_h5ad(PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad", backed="r")

pairwise_comparisons = [("TNBC", "ER+"), ("HER2+", "ER+"), ("TNBC", "HER2+")]
all_sccoda_results_2 = {}

for group_a, group_b in pairwise_comparisons:
    comparison_name = f"{group_a}_vs_{group_b}"
    obs2_sub = adata2.obs[adata2.obs["subtype"].isin([group_a, group_b])].copy()
    obs2_sub["subtype"] = obs2_sub["subtype"].astype(str)  # drop unused categorical levels

    comp_adata2 = build_composition_anndata(obs2_sub, sample_col="orig.ident", condition_col="subtype")
    print(f"\n{comparison_name}: {comp_adata2.n_obs} samples x {comp_adata2.n_vars} cell types")

    model2 = scc_ana.CompositionalAnalysis(
        comp_adata2, formula="subtype", reference_cell_type="PVL"
    )
    result2 = model2.sample_hmc()

    print(f"=== GSE176078: {comparison_name} — scCODA results ===")
    result2.summary()

    effects2 = result2.effect_df
    effects2.to_csv(RESULTS_DIR / f"GSE176078_scCODA_{comparison_name}_effects.csv")
    all_sccoda_results_2[comparison_name] = result2

    credible2 = result2.credible_effects()
    print(f"\nCredible effects ({comparison_name}):")
    print(credible2)
    gc.collect()

print("\nGSE176078 scCODA complete")